# Data Sharing & Listings Audit Notebook

Portable audit of everything that moves data across the account boundary through Snowflake's
sharing features: outbound shares, inbound shares, listings, reader accounts, and replication.
Written for a regulated / confidential-data environment. Run top to bottom. It surfaces:

1. **Who is allowed to share** — roles holding `CREATE SHARE`, `CREATE LISTING`, `IMPORT SHARE`, and related privileges
2. **Outbound share inventory** — every share, its consumers, and whether any consumers are reader accounts
3. **Share contents** — exactly which objects each share exposes
4. **Sensitive exposure** — shared objects containing columns that look sensitive by name
5. **Stale shares** — shares with no consumers, or no objects
6. **Inbound shares** — data coming in, and who can access it
7. **Listings** — provider and consumer listings, and their usage telemetry where available
8. **Reader (managed) accounts and replication/failover groups** — other data-movement paths
9. **Change history** — recent create/alter/grant activity on shares and listings
10. **A rolled-up review checklist**

### Prerequisites
- `ACCOUNTADMIN` (or a role with `IMPORTED PRIVILEGES` on `SNOWFLAKE` plus the privileges to run
  `SHOW SHARES`, `SHOW LISTINGS`, and `SHOW MANAGED ACCOUNTS`).
- `SNOWFLAKE.DATA_SHARING_USAGE` views (Section 7) are provider-side and may need to be
  granted separately; the cell notes what to do if it errors.
- Nothing here is destructive. Every cell reads.

### A note on view availability
Sharing-related `ACCOUNT_USAGE` and `DATA_SHARING_USAGE` views have changed across Snowflake
releases and vary by edition. Where a `SHOW` command is the most reliable source, this
notebook uses it. If a view errors in this environment, fall back to the `SHOW` equivalent
and log the difference — it's also worth noting in the release-note tracker.


In [ ]:
-- ============================================================
-- PARAMETERS
-- ============================================================
SET lookback_days = 90;
SET sharing_privileges = 'CREATE SHARE,IMPORT SHARE,CREATE LISTING,MANAGE LISTING AUTO FULFILLMENT,ATTACH LISTING,CREATE DATA EXCHANGE LISTING,IMPORT DATA EXCHANGE';

SELECT $lookback_days AS lookback_days, $sharing_privileges AS sharing_privileges;


## 1. Who is allowed to share

The privileges that let someone create a share or listing, or import someone else's, are
account-level grants. In a confidential-data environment this should be a very short list of
roles, ideally not including anything broadly assigned to analysts or engineers.


In [ ]:
-- Roles holding sharing-related account privileges
SELECT
    grantee_name AS role_name,
    privilege,
    grant_option,
    granted_by,
    created_on
FROM snowflake.account_usage.grants_to_roles
WHERE deleted_on IS NULL
  AND granted_on = 'ACCOUNT'
  AND privilege IN (SELECT TRIM(value) FROM TABLE(SPLIT_TO_TABLE($sharing_privileges, ',')))
ORDER BY privilege, role_name;


In [ ]:
-- How many users can effectively exercise those privileges (via direct role membership)
WITH sharing_roles AS (
    SELECT DISTINCT grantee_name AS role_name
    FROM snowflake.account_usage.grants_to_roles
    WHERE deleted_on IS NULL
      AND granted_on = 'ACCOUNT'
      AND privilege IN (SELECT TRIM(value) FROM TABLE(SPLIT_TO_TABLE($sharing_privileges, ',')))
)
SELECT
    gu.role AS role_name,
    COUNT(DISTINCT gu.grantee_name) AS users_holding_role,
    ARRAY_AGG(DISTINCT gu.grantee_name) WITHIN GROUP (ORDER BY gu.grantee_name) AS users
FROM snowflake.account_usage.grants_to_users gu
JOIN sharing_roles sr ON gu.role = sr.role_name
WHERE gu.deleted_on IS NULL
GROUP BY gu.role
ORDER BY users_holding_role DESC;


**Note:** this only counts direct role membership. `ACCOUNTADMIN` inherits everything, so
also review who holds `ACCOUNTADMIN` (see the RBAC audit notebook, Section 2). Roles that
inherit a sharing role through the role hierarchy aren't expanded here.


## 2. Outbound share inventory

Every outbound share is a sanctioned path for data to leave the account. Each one needs an
owner, a business reason, and a known consumer list.


In [ ]:
SHOW SHARES;


In [ ]:
-- Outbound shares only, with consumer accounts (the "to" column) and ownership
SELECT
    "created_on"           AS created_on,
    "name"                 AS share_name,
    "database_name"        AS source_database,
    "to"                   AS consumer_accounts,
    "owner"                AS owner_role,
    "comment"              AS comment,
    "listing_global_name"  AS listing_global_name
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
WHERE "kind" = 'OUTBOUND'
ORDER BY "created_on" DESC;


**Watch for:** shares with an empty `comment` (no documented purpose), an `owner_role` that
is an individual's personal role or `ACCOUNTADMIN`, consumer lists containing accounts nobody
on the team recognizes, and a non-null `listing_global_name` (the share is backing a listing,
which may make it visible beyond the named consumers).


## 3. Share contents and consumers (Python)

`SHOW GRANTS TO SHARE` shows the objects in each share; `SHOW GRANTS OF SHARE` shows the
consumer accounts that have actually been granted access. Looping per share is fine here
since outbound shares are usually few in number.


In [ ]:
# Objects and consumers for every outbound share.
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()

shares = session.sql("SHOW SHARES").collect()
outbound = [s["name"] for s in shares if s["kind"] == "OUTBOUND"]

object_rows, consumer_rows = [], []
for share_name in outbound:
    try:
        for g in session.sql(f'SHOW GRANTS TO SHARE "{share_name}"').collect():
            d = g.as_dict()
            object_rows.append({
                "share_name": share_name,
                "privilege": d.get("privilege"),
                "object_type": d.get("granted_on"),
                "object_name": d.get("name"),
                "granted_by": d.get("granted_by"),
                "created_on": d.get("created_on"),
            })
    except Exception as e:
        object_rows.append({"share_name": share_name, "privilege": "ERROR", "object_type": None,
                            "object_name": str(e), "granted_by": None, "created_on": None})
    try:
        for g in session.sql(f'SHOW GRANTS OF SHARE "{share_name}"').collect():
            d = g.as_dict()
            consumer_rows.append({
                "share_name": share_name,
                "consumer": d.get("grantee_name"),
                "granted_to": d.get("granted_to"),
                "granted_by": d.get("granted_by"),
                "created_on": d.get("created_on"),
            })
    except Exception as e:
        consumer_rows.append({"share_name": share_name, "consumer": f"ERROR: {e}",
                              "granted_to": None, "granted_by": None, "created_on": None})

share_objects_df = pd.DataFrame(object_rows)
share_consumers_df = pd.DataFrame(consumer_rows)
share_objects_df


In [ ]:
# Consumers per share. A share with zero consumers is a cleanup candidate (Section 5).
share_consumers_df


## 4. Sensitive exposure through shares

Cross-references every table/view exposed through an outbound share against columns whose names
look sensitive. This is a name-matching heuristic (same caveat as the security posture
notebook): it triages where to look first, it isn't data classification.


In [ ]:
# Sensitive-looking columns inside shared objects.
SENSITIVE_PATTERNS = ['SSN', 'SOCIAL_SEC', 'PATIENT', 'MRN', 'DOB', 'BIRTH', 'DIAGNOS', 'EMAIL',
                      'PHONE', 'ADDRESS', 'INSURANCE', 'MEDICAL', 'HEALTH', 'GENDER', 'RACE',
                      'ZIP', 'CREDIT_CARD']

if share_objects_df.empty:
    print("No outbound share objects found.")
else:
    tabular = share_objects_df[share_objects_df["object_type"].isin(["TABLE", "VIEW", "MATERIALIZED_VIEW", "EXTERNAL_TABLE", "DYNAMIC_TABLE"])].copy()
    parts = tabular["object_name"].str.replace('"', '', regex=False).str.split(".", n=2, expand=True)
    tabular["db"], tabular["schema"], tabular["obj"] = parts[0], parts[1], parts[2]

    like_clause = " OR ".join([f"column_name ILIKE '%{p}%'" for p in SENSITIVE_PATTERNS])
    cols_df = session.sql(f'''
        SELECT table_catalog AS db, table_schema AS schema, table_name AS obj, column_name, data_type
        FROM snowflake.account_usage.columns
        WHERE deleted IS NULL AND ({like_clause})
    ''').to_pandas()
    cols_df.columns = [c.lower() for c in cols_df.columns]

    exposure = tabular.merge(cols_df, on=["db", "schema", "obj"], how="inner")
    exposure = exposure[["share_name", "db", "schema", "obj", "column_name", "data_type"]].sort_values(
        ["share_name", "db", "schema", "obj", "column_name"])
    print(f"{len(exposure)} sensitive-looking columns exposed across {exposure['share_name'].nunique()} share(s)")
exposure if not share_objects_df.empty else None


For each finding: is the column masked, or is the shared object a secure view that
excludes/de-identifies it? Confirm with the data owner and the privacy/compliance team before
concluding anything — and check whether the share contains base tables directly (broader
exposure) versus a curated secure view (narrower, easier to govern).


## 5. Stale shares

Shares with no consumers, no objects, or no recent purpose are candidates for cleanup. A
forgotten share with live consumers attached is the worse case — data still flowing out with
nobody watching it.


In [ ]:
# Shares with no objects, and shares with no consumers.
no_objects = sorted(set(outbound) - set(share_objects_df["share_name"].unique())) if not share_objects_df.empty else outbound
consumer_names = set(share_consumers_df["share_name"].unique()) if not share_consumers_df.empty else set()
no_consumers = sorted(set(outbound) - consumer_names)

print("Outbound shares with NO objects:", no_objects or "none")
print("Outbound shares with NO consumers:", no_consumers or "none")


## 6. Inbound shares

Data coming *into* the account is a governance question too: whose data is it, who can query
it, and does anything downstream copy it into internal tables (at which point the provider's
controls no longer apply)?


In [ ]:
-- Inbound shares and the databases created from them
SELECT
    "created_on"          AS created_on,
    "owner_account"       AS provider_account,
    "name"                AS share_name,
    "database_name"       AS local_database,
    "comment"             AS comment
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
WHERE "kind" = 'INBOUND'
ORDER BY "created_on" DESC;


In [ ]:
-- Who holds IMPORTED PRIVILEGES on databases created from shares
SELECT
    grantee_name AS role_name,
    privilege,
    name         AS database_name,
    granted_by,
    created_on
FROM snowflake.account_usage.grants_to_roles
WHERE deleted_on IS NULL
  AND granted_on = 'DATABASE'
  AND privilege = 'IMPORTED PRIVILEGES'
ORDER BY database_name, role_name;


## 7. Listings

Listings (private, marketplace, or data exchange) package shares for discovery and can
broaden who is able to request access. Provider-side and consumer-side are separate `SHOW`
commands.


In [ ]:
-- Listings this account has published (provider side)
SHOW LISTINGS;


In [ ]:
-- Listings this account can see or has accessed (consumer side)
SHOW AVAILABLE LISTINGS IS_IMPORTED = TRUE;


**Watch for:** listings with public/marketplace visibility rather than targeted private
sharing, listings not tied to a documented business purpose, and any listing whose state is
`PUBLISHED` that nobody on the platform team recognizes.


In [ ]:
-- Provider-side listing consumption/access telemetry (may require a grant on
-- SNOWFLAKE.DATA_SHARING_USAGE; if it errors, ask the account admin for access or skip)
SELECT *
FROM snowflake.data_sharing_usage.listing_access_history
WHERE event_date >= DATEADD(day, -$lookback_days, CURRENT_DATE())
ORDER BY event_date DESC
LIMIT 200;


## 8. Other data-movement paths

Reader accounts (Snowflake-managed accounts you create for consumers who don't have their own)
and replication/failover groups both move data across boundaries without being called
"shares," so they're easy to miss in a sharing review.


In [ ]:
-- Reader (managed) accounts created by this account
SHOW MANAGED ACCOUNTS;


In [ ]:
-- Replication and failover groups: data replicated to other accounts/regions
SHOW REPLICATION GROUPS;


In [ ]:
SHOW FAILOVER GROUPS;


In [ ]:
-- Cross-region / cross-cloud data transfer volume (a cost and an egress signal)
SELECT
    source_cloud, source_region, target_cloud, target_region, transfer_type,
    ROUND(SUM(bytes_transferred) / POWER(1024,3), 2) AS gb_transferred
FROM snowflake.account_usage.data_transfer_history
WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
GROUP BY source_cloud, source_region, target_cloud, target_region, transfer_type
ORDER BY gb_transferred DESC;


## 9. Change history

Who created or modified shares and listings recently, and when. Useful for tying an
unexpected share back to a person and a date, and for the ongoing-monitoring version of this
audit.


In [ ]:
-- Recent share / listing DDL and grants
SELECT
    start_time,
    user_name,
    role_name,
    query_type,
    LEFT(query_text, 300) AS statement
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
  AND execution_status = 'SUCCESS'
  AND (
        query_text ILIKE 'CREATE%SHARE%'
     OR query_text ILIKE 'ALTER%SHARE%'
     OR query_text ILIKE 'DROP%SHARE%'
     OR query_text ILIKE 'GRANT%TO SHARE%'
     OR query_text ILIKE 'REVOKE%FROM SHARE%'
     OR query_text ILIKE '%LISTING%' AND query_type IN ('CREATE', 'ALTER', 'DROP', 'GRANT', 'REVOKE', 'UNKNOWN')
     OR query_text ILIKE 'CREATE%MANAGED ACCOUNT%'
      )
ORDER BY start_time DESC
LIMIT 300;


## 10. Review checklist

Fill in from the results above.

| Area | Finding | Risk if unaddressed | Status |
|---|---|---|---|
| Sharing privileges limited to a short, named list of roles | *Section 1* | Anyone with the privilege can send data out of the account | |
| Every outbound share has an owner, purpose, and comment | *Section 2* | Undocumented egress path | |
| Consumer accounts on every share are recognized and approved | *Sections 2-3* | Data exposed to unknown parties | |
| Shares expose curated secure views, not raw base tables | *Section 3* | Broader exposure than needed | |
| Sensitive-looking columns in shared objects reviewed | *Section 4* | Confidential data shared unmasked | |
| Stale shares removed | *Section 5* | Forgotten live egress | |
| Inbound share access scoped, and no uncontrolled copies downstream | *Section 6* | Provider controls bypassed | |
| Listings reviewed for visibility and purpose | *Section 7* | Wider access than intended | |
| Reader accounts and replication groups inventoried | *Section 8* | Data-movement paths outside the sharing review | |
| Change history reviewed; ongoing alerting considered | *Section 9* | Drift goes unnoticed | |


### Notes, caveats, and next steps

- **Consumer-side visibility stops at the account boundary.** You can see who a share is
  granted to, but not what the consumer does with the data afterward. Contract and BAA-type
  terms are what govern that, not anything in this notebook.
- **Re-run as a recurring check.** Sections 2, 5, and 9 are the ones worth scheduling: new
  shares, new consumers, and share-related DDL are exactly the events a regulated environment
  wants to notice quickly rather than at the next audit.
- **Pair with the security posture notebook.** Its Section 6 (masking coverage) and this
  notebook's Section 4 overlap deliberately: the same unmasked sensitive column is a bigger
  finding if it is also exposed through a share.
